# LightGBM — Prédiction de la consommation électrique (features physiques agrégées)
**Sous-ensemble :** Maisons individuelles plein pied, chauffage électrique, sans VE/piscine/PV
**Cibles :** électricité totale + par usage (chauffage, clim, eau chaude) — cible principale : `total`
**Split :** Stratification par zones ASHRAE IECC 2004
**Différence avec `lgbm_electricity` :** le bloc enveloppe/géométrie (~30 colonnes brutes) est remplacé par **5 agrégats physiques** (UA, H_ve, C, A_solaire, compacité). Le reste des variables (HVAC, occupants, climat, électroménager) est conservé.
**Différence avec `lgbm_electricity_usages` :** mêmes 4 cibles, mais sur le sous-ensemble filtré et avec les agrégats. L'analyse détaillée (erreurs par zone, importance, courbe du coude) reste sur la cible `total`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED      = 42
LGBM_BASE = dict(random_state=SEED, n_jobs=-1, verbose=-1)

# cibles électricité par usage (nom court -> colonne brute)
TARGETS = {
    'total'      : 'out.electricity.total.energy_consumption..kwh',
    'chauffage'  : 'out.electricity.heating.energy_consumption..kwh',
    'clim'       : 'out.electricity.cooling.energy_consumption..kwh',
    'eau_chaude' : 'out.electricity.hot_water.energy_consumption..kwh',
}
TARGET = 'total'   # cible principale : pipeline détaillé + Optuna

def to_numpy_dtypes(df):
    for col in df.columns:
        dt = df[col].dtype
        if hasattr(dt, 'numpy_dtype'):
            df[col] = df[col].astype(dt.numpy_dtype)
        elif hasattr(dt, 'pyarrow_dtype'):
            df[col] = df[col].astype(str(dt.pyarrow_dtype))
    return df

def metrics(y_true, y_pred, label=''):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    if label:
        print(f'{label:12} | R²={r2:.4f} | RMSE={rmse:8.0f} kWh | MAE={mae:7.0f} kWh')
    return r2, rmse, mae

## Construction des 5 agrégats physiques
Voir le document `reports/modele_enveloppe_thermique_5features.docx` pour les définitions et formules.
Constantes de conversion : `R_SI = R_US × 0.1761`, `U_SI = U_US × 5.678`. Films de surface ≈ 0.85 (US).

In [ ]:
import re

FT2   = 0.0929     # ft² -> m²
RUS   = 0.1761     # R US -> R SI (m²K/W)
UUS   = 5.678      # U US -> U SI (W/m²K)
RFILM = 0.85       # films de surface (US), évite les U infinis
ISO   = dict(leger=110, moyen=165, lourd=260, tres_lourd=370)   # kJ/m²K (classes ISO 13790)

def _U(Rus):
    return 1.0 / ((np.maximum(Rus, 0) + RFILM) * RUS)

def _poids(az):
    az %= 360
    if 135 <= az <= 225:      # Sud
        return 1.0
    if az < 45 or az >= 315:  # Nord
        return 0.3
    return 0.6                 # Est / Ouest

def build_aggregates(Xs, Rs):
    """Xs = features encodées (X), Rs = colonnes out.params + matériaux (raw). Index alignés."""
    A_wall  = Rs['out.params.wall_area_above_grade_exterior..ft2'].values * FT2
    A_roof  = Rs['out.params.roof_area..ft2'].values                     * FT2
    A_floor = Rs['out.params.floor_area_lighting..ft2'].values           * FT2
    A_win   = Rs['out.params.window_area..ft2'].values                   * FT2
    A_door  = Rs['out.params.door_area..ft2'].values                     * FT2

    # 1) UA — déperditions par transmission (toiture routée via attic pour éviter le plafond fantôme)
    is_attic = Rs['in.geometry_attic_type'].astype(str).isin(['Vented Attic', 'Unvented Attic']).values
    R_top    = np.where(is_attic, Xs['in.insulation_ceiling'].values, Xs['in.insulation_roof'].values)
    UA = (_U(Xs['in.insulation_wall'].values) * A_wall
          + _U(R_top)                          * A_roof
          + _U(Xs['in.insulation_floor'].values) * A_floor
          + Xs['in.window_ufactor'].values * UUS * A_win
          + 1.14                                 * A_door)

    # 2) H_ve — déperditions par renouvellement d'air
    V    = A_floor * 2.5
    H_ve = 0.34 * (Xs['in.air_leakage_to_outside_ach50'].values / 20.0) * V

    # 3) C — inertie thermique (classe ISO 13790 × surface plancher)
    wt  = Rs['in.geometry_wall_type'].astype(str)
    fin = Rs['in.geometry_wall_exterior_finish'].astype(str)
    cls = np.select(
        [wt.str.contains('Concrete').values, fin.str.contains('Brick').values,
         (wt == 'Wood Frame').values, (wt == 'Steel Frame').values],
        [ISO['tres_lourd'], ISO['lourd'], ISO['leger'], ISO['moyen']], default=ISO['moyen'])
    C = cls * A_floor

    # 4) A_solaire — surface solaire équivalente (pondérée par orientation de chaque façade)
    shgc = Xs['in.window_shgc'].values
    ang  = np.degrees(np.arctan2(Xs['in.orientation_sin'].values,
                                 Xs['in.orientation_cos'].values)) % 360
    wa   = Rs['in.window_areas'].astype(str).values
    Asol = np.zeros(len(Xs))
    for i, (s, a, sh, aw) in enumerate(zip(wa, ang, shgc, A_win)):
        d = {'F': 0., 'B': 0., 'L': 0., 'R': 0.}
        for m in re.finditer(r'([FBLR])(\d+)', s):
            d[m.group(1)] = float(m.group(2))
        tot  = sum(d.values()) or 1.0
        dirs = {'F': a, 'R': a + 90, 'B': a + 180, 'L': a + 270}
        Asol[i] = sh * 0.9 * aw * sum((d[k] / tot) * _poids(dirs[k]) for k in d)

    # 5) compacité — surface déperditive / volume
    comp = (A_wall + A_roof + A_floor + A_win + A_door) / np.maximum(V, 1)

    return pd.DataFrame({'UA': UA, 'H_ve': H_ve, 'C': C, 'A_solaire': Asol, 'compacite': comp},
                        index=Xs.index)

In [ ]:
def build_hvac_dse(Xs):
    """DSE — rendement de distribution des gaines (ASHRAE 152 simplifié, cf. ResStock Tech. Ref. Guide).

    Le guide : les pertes (fuite + conduction) ne comptent que pour la part des gaines HORS volume
    chauffé ; et pour les maisons à 1 étage, 100 % de la surface des gaines est hors volume chauffé
    -> f_loc binaire (0 si gaines en volume conditionné, 1 sinon).

    duct_location_int : 0 None, 1 Living Space, 4 Heated Basement  -> conditionné (DSE = 1)
                        2 Attic, 3 Crawlspace, 5 Unheated Basement, 6 Garage -> non conditionné
    """
    loc  = Xs['in.duct_location_int'].values     # localisation encodée
    leak = Xs['in.duct_leakage'].values          # fuite vers l'extérieur : 0 / 0,1 / 0,2 / 0,3
    rins = Xs['in.duct_insulation'].values       # R des gaines : 0 / 4 / 6 / 8

    conditionne      = np.isin(loc, [0, 1, 4])                 # None / Living Space / Heated Basement
    perte_fuite      = leak
    perte_conduction = 0.10 / (1.0 + rins / 4.0)               # R0 -> 0,10  ...  R8 -> 0,033
    dse = np.where(conditionne, 1.0,
                   np.clip(1.0 - (perte_fuite + perte_conduction), 0.5, 1.0))
    return pd.Series(dse, index=Xs.index, name='DSE')

In [ ]:
ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW       = ROOT / 'data' / 'raw'

X    = pd.read_parquet(DATA_PROCESSED / 'X.parquet')
meta = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet')

# colonnes nécessaires au calcul des agrégats, prises dans le fichier brut
RAW_COLS = ['out.params.wall_area_above_grade_exterior..ft2', 'out.params.roof_area..ft2',
            'out.params.floor_area_lighting..ft2', 'out.params.window_area..ft2',
            'out.params.door_area..ft2', 'in.geometry_attic_type', 'in.geometry_wall_type',
            'in.geometry_wall_exterior_finish', 'in.window_areas']
raw = pd.read_parquet(DATA_RAW / 'upgrade0.parquet', columns=RAW_COLS)

# cibles par usage : lues dans le fichier brut (pas dans Y.parquet)
Yt = pd.read_parquet(DATA_RAW / 'upgrade0.parquet', columns=list(TARGETS.values()))
Yt = Yt.rename(columns={v: k for k, v in TARGETS.items()})

# bloc enveloppe/géométrie remplacé par les 5 agrégats -> on le retire
DROP = [c for c in X.columns if any(k in c for k in [
    'insulation_', 'slab_', 'window_ufactor', 'window_shgc', 'window_front', 'air_leakage',
    'wall_exterior_finish_r', 'wall_finish_', 'roof_material', 'geometry_floor_area',
    'geometry_stories', 'geometry_foundation_type', 'geometry_garage', 'orientation_',
    'horiz_loc_', 'neighbor_'])]

# socio-éco : on ne garde que income + federal_poverty_level (les autres < 0,1 % d'importance)
DROP_SOCIO = ['in.area_median_income', 'in.state_metro_median_income', 'in.tenure',
              'in.household_has_tribal_persons', 'in.aiannh_area']

# chauffe-eau (DHW) : on ne garde que efficiency + location_semi_conditioned
# (les 8 autres < 0,02 % sur total ET eau_chaude ; combustibles quasi constants car subset élec.)
DROP_DHW = ['in.water_heater_in_unit', 'in.water_heater_technology_indirect',
            'in.water_heater_technology_storage', 'in.water_heater_technology_tankless',
            'in.water_heater_fuel_fuel_oil', 'in.water_heater_fuel_natural_gas',
            'in.water_heater_fuel_propane', 'in.water_heater_location_unconditioned']

# usages "niveau 3" : ~0 % d'importance sur les 4 cibles -> retirés.
# (flags de présence has_*, bloc piscine constant par le filtre, bloc gaz, hot_tub, ceiling_fan, well_pump)
# NB : has_dishwasher conservé volontairement (bloc dishwasher gardé), dishwasher_kwh conservé aussi.
DROP_USAGES = ['in.hot_tub_electric', 'in.has_hot_tub', 'in.misc_hot_tub_gas',
               'in.has_pool', 'in.pool_heater_present', 'in.pool_heater_electric', 'in.pool_heater_gas',
               'in.misc_gas_fireplace_present', 'in.misc_gas_grill_present', 'in.misc_gas_lighting_present',
               'in.has_well_pump', 'in.has_ceiling_fan', 'in.ceiling_fan_used',
               'in.clothes_dryer_gas', 'in.clothes_dryer_electric', 'in.clothes_dryer_has',
               'in.clothes_washer_has', 'in.refrigerator_has', 'in.refrigerator_usage_level',
               'in.misc_extra_refrigerator_has']

# gaines HVAC : les 4 features de distribution sont fondues dans l'agrégat DSE (cf. build_hvac_dse)
DROP_HVAC_DUCTS = ['in.hvac_has_ducts', 'in.duct_location_int', 'in.duct_leakage', 'in.duct_insulation']

DROP = DROP + [c for c in DROP_SOCIO + DROP_DHW + DROP_USAGES + DROP_HVAC_DUCTS if c in X.columns]

KEEP = [c for c in X.columns if c not in DROP]

mask = (
    (meta['in.geometry_building_type_recs'] == 'Single-Family Detached') &
    (X['in.geometry_stories'] == 1)                                      &
    (meta['in.heating_fuel']  == 'Electricity')                          &
    (X['in.electric_vehicle_charger'] == 0)                              &
    (X['in.has_pool'] == 0)                                              &
    (X['in.has_pv']   == 0)
).values

# agrégats enveloppe (5) + DSE gaines (1) + variables conservées
agg    = build_aggregates(X[mask], raw[mask]).reset_index(drop=True)
dse    = build_hvac_dse(X[mask]).reset_index(drop=True)
keep   = X[mask][KEEP].reset_index(drop=True)
X_sub  = to_numpy_dtypes(pd.concat([agg, dse, keep], axis=1))
Yt_sub = to_numpy_dtypes(Yt[mask].reset_index(drop=True).copy())
ashrae_sub = meta[mask]['in.ashrae_iecc_climate_zone_2004'].astype(str).reset_index(drop=True)

# export pour le réseau time series (notebooks/06_timeseries/timeseries_net) :
# les mêmes 47 features, indexées par bldg_id pour être jointes aux profils horaires
X_sub.set_index(pd.Index(meta[mask]['bldg_id'].values, name='bldg_id')) \
     .to_parquet(DATA_PROCESSED / 'X_47features.parquet')

print(f'{len(X_sub):,} logements | {X_sub.shape[1]} features '
      f'({agg.shape[1]} agrégats enveloppe + 1 DSE + {len(KEEP)} autres) | {len(TARGETS)} cibles '
      f'| {ashrae_sub.nunique()} zones ASHRAE')
print('Agrégats :', list(agg.columns) + ['DSE'])
print('Socio-éco conservées   :', [c for c in ['in.income', 'in.federal_poverty_level'] if c in KEEP])
print('Chauffe-eau conservés  :', [c for c in KEEP if 'water_heater' in c])
print(f'DSE — moyenne={dse.mean():.3f} | conditionné (DSE=1)={100*(dse==1).mean():.0f}% '
      f'| min={dse.min():.2f}')
print('Export -> X_47features.parquet')
print('\nPart de logements avec conso > 0 par usage :')
for k in TARGETS:
    print(f'  {k:12} : {(Yt_sub[k] > 0).mean()*100:5.1f}%  (moyenne {Yt_sub[k].mean():.0f} kWh)')

In [ ]:
counts_z = ashrae_sub.value_counts()
rare_z   = counts_z[counts_z < 5].index
strat    = ashrae_sub.where(~ashrae_sub.isin(rare_z), other='RARE')

# split unique (stratifié ASHRAE), partagé par toutes les cibles
X_tv, X_test, Yt_tv, Yt_test, strat_tv, _, ash_tv, ash_test = train_test_split(
    X_sub, Yt_sub, strat, ashrae_sub,
    test_size=0.2, random_state=SEED, stratify=strat
)

X_train, X_val, Yt_train, Yt_val, strat_tr, strat_val, ash_train, ash_val = train_test_split(
    X_tv, Yt_tv, strat_tv, ash_tv,
    test_size=0.2, random_state=SEED, stratify=strat_tv
)

X_trainval   = X_tv
Yt_trainval  = Yt_tv
ash_test_arr = ash_test.values

# cible principale ('total') : séries utilisées par le pipeline détaillé
Y_train, Y_val, Y_test = Yt_train[TARGET], Yt_val[TARGET], Yt_test[TARGET]
Y_trainval = Yt_trainval[TARGET]

print(f'Train : {len(X_train):,} | Val : {len(X_val):,} | Test : {len(X_test):,}')
print(f'Y_train ({TARGET})  mean={Y_train.mean():.0f} kWh | median={Y_train.median():.0f} kWh')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(Y_train, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(Y_train.mean(),   color='red',    ls='--', lw=1.5, label=f'Moyenne  {Y_train.mean():.0f} kWh')
axes[0].axvline(Y_train.median(), color='orange', ls='--', lw=1.5, label=f'Médiane  {Y_train.median():.0f} kWh')
axes[0].set(title='Distribution de Y (linéaire)', xlabel='kWh/an', ylabel='Logements')
axes[0].legend()

axes[1].hist(Y_train, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[1].set_yscale('log')
axes[1].set(title='Distribution de Y (échelle log)', xlabel='kWh/an', ylabel='Logements (log)')

plt.suptitle(f'Consommation électrique — train : {len(Y_train):,} logements | skewness={Y_train.skew():.2f}', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
def objective(trial):
    params = {
        'n_estimators'     : 3000,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 120),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 500),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'subsample'        : trial.suggest_float('subsample', 0.4, 1.0),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 0.1, 100.0, log=True),
        'reg_alpha'        : trial.suggest_float('reg_alpha',  0.01,  20.0, log=True),
        **LGBM_BASE,
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(X_train, Y_train, eval_set=[(X_val, Y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    return np.sqrt(mean_squared_error(Y_val, m.predict(X_val)))

study = optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'Best val RMSE : {study.best_value:.0f} kWh')
for k, v in study.best_params.items():
    print(f'  {k:25} : {v}')

In [ ]:
final_params = {**study.best_params, 'n_estimators': 10_000, **LGBM_BASE}

model = lgb.LGBMRegressor(**final_params)
model.fit(
    X_trainval, Y_trainval,
    eval_set  = [(X_test, Y_test)],
    callbacks = [lgb.early_stopping(100, verbose=False)],
)

pred_trainval = model.predict(X_trainval)
pred_test     = model.predict(X_test)

print(f'Arbres : {model.best_iteration_:,}\n')
r2_tv, rmse_tv, mae_tv = metrics(Y_trainval, pred_trainval, 'Train+Val')
r2_te, rmse_te, mae_te = metrics(Y_test,     pred_test,     'Test')
print(f'\nOverfit gap : {r2_tv - r2_te:.4f} | RMSE% : {rmse_te / Y_test.mean() * 100:.1f}%')

In [ ]:
zones_uniq = sorted(set(ash_test_arr))
cmap       = plt.colormaps['tab20']
z2c        = {z: cmap(i / max(len(zones_uniq) - 1, 1)) for i, z in enumerate(zones_uniq)}

fig, ax = plt.subplots(figsize=(9, 8))

for zone in zones_uniq:
    mask_z = ash_test_arr == zone
    ax.scatter(
        Y_test.values[mask_z],
        pred_test[mask_z],
        color=z2c[zone],
        label=zone,
        alpha=0.4,
        s=8,
        edgecolors='none',
    )

lo = min(Y_test.min(), pred_test.min()) * 0.95
hi = max(Y_test.max(), pred_test.max()) * 1.02
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.2, label='Prédiction parfaite (y=x)')

ax.text(0.04, 0.96, f'R² = {r2_te:.4f}',
        transform=ax.transAxes, fontsize=13, va='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

ax.set_xlabel('Consommation réelle (kWh/an)', fontsize=12)
ax.set_ylabel('Consommation prédite (kWh/an)', fontsize=12)
ax.set_title('Réel vs Prédit — coloré par zone ASHRAE', fontsize=13)
ax.legend(title='Zone ASHRAE', bbox_to_anchor=(1.01, 1), loc='upper left',
          fontsize=8, markerscale=2)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
df_err = pd.DataFrame({
    'zone'    : ash_test_arr,
    'y_true'  : Y_test.values,
    'y_pred'  : pred_test,
    'err_abs' : np.abs(Y_test.values - pred_test),
    'err_rel' : np.abs(Y_test.values - pred_test) / Y_test.values * 100,
    'bias'    : pred_test - Y_test.values,
})

global_rmse = np.sqrt((df_err['err_abs'] ** 2).mean())
global_mae  = df_err['err_abs'].mean()
global_mape = df_err['err_rel'].mean()

stats_zone = df_err.groupby('zone').agg(
    n        = ('y_true', 'count'),
    conso_med= ('y_true', 'median'),
    rmse     = ('err_abs', lambda x: np.sqrt((x**2).mean())),
    mae      = ('err_abs', 'mean'),
    mape     = ('err_rel', 'mean'),
    biais    = ('bias', 'mean'),
).round(0).sort_values('rmse')

print(f'Global  RMSE={global_rmse:.0f} kWh | MAE={global_mae:.0f} kWh | MAPE={global_mape:.1f}%\n')
print(stats_zone.to_string())

zones_ord  = stats_zone.index.tolist()
bar_colors = [z2c[z] for z in zones_ord]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, gval, ylabel, title in [
    (axes[0], 'rmse', global_rmse, 'RMSE (kWh)', 'RMSE par zone ASHRAE'),
    (axes[1], 'mae',  global_mae,  'MAE (kWh)',  'MAE par zone ASHRAE'),
    (axes[2], 'mape', global_mape, 'MAPE (%)',   'MAPE par zone ASHRAE'),
]:
    ax.bar(zones_ord, stats_zone.loc[zones_ord, col], color=bar_colors, edgecolor='white')
    ax.axhline(gval, color='black', ls='--', lw=1.2, label=f'Global ({gval:.0f})')
    ax.set(title=title, xlabel='Zone ASHRAE', ylabel=ylabel)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Analyse des erreurs par zone ASHRAE', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.Series(model.feature_importances_, index=X_trainval.columns).sort_values(ascending=False)
AGG = ['UA', 'H_ve', 'C', 'A_solaire', 'compacite', 'DSE']   # 5 agrégats enveloppe + DSE gaines

# ── Graphe : TOUS les attributs classés par importance (agrégats en rouge) ──
ordre  = importance.sort_values()
colors = ['crimson' if f in AGG else 'steelblue' for f in ordre.index]

fig, ax = plt.subplots(figsize=(10, max(8, len(ordre) * 0.22)))
ax.barh(ordre.index, ordre.values, color=colors)
ax.set(title=f'Feature importance — {len(importance)} attributs (rouge = agrégats physiques)',
       xlabel='Importance (gain LightGBM)')
ax.tick_params(axis='y', labelsize=7)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# ── Tableau : TOUS les attributs classés par importance ──
rang = pd.DataFrame({
    'importance': importance.round(0).astype(int),
    'part_%'    : (importance / importance.sum() * 100).round(2),
    'agrégat'   : ['oui' if f in AGG else '' for f in importance.index],
})
rang.index.name = 'feature'
rang.insert(0, 'rang', range(1, len(rang) + 1))

pd.set_option('display.max_rows', None)
print(f'=== {len(rang)} attributs classés par importance ===\n')
print(rang.to_string())
pd.reset_option('display.max_rows')

print(f"\nPart cumulée des {len(AGG)} agrégats physiques : "
      f"{rang.loc[rang['agrégat'] == 'oui', 'part_%'].sum():.1f} %")

In [ ]:
X_num = X_train.select_dtypes(include='number')
corr  = X_num.corrwith(Y_train).dropna().sort_values(key=abs, ascending=False)

print('=== Top 20 features — corrélation Pearson avec la consommation électrique ===\n')
print(corr.head(20).round(3).to_string())

top20 = corr.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
colors_c = ['tomato' if v < 0 else 'steelblue' for v in top20.values[::-1]]
ax.barh(top20.index[::-1], top20.values[::-1], color=colors_c)
ax.axvline(0, color='black', lw=0.8)
ax.set(title='Top 20 features — Corrélation Pearson avec la consommation électrique',
       xlabel='Corrélation de Pearson')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
feat_order = importance.index.tolist()
n_total    = len(feat_order)
K_values   = [5, 10, 20, 30, 50, n_total]
cap_iter   = model.best_iteration_ + 100

r2_by_k   = {}
rmse_by_k = {}

for K in K_values:
    top_feats = feat_order[:K]
    m = lgb.LGBMRegressor(
        n_estimators=cap_iter,
        **{k: v for k, v in final_params.items() if k != 'n_estimators'},
    )
    m.fit(
        X_trainval[top_feats], Y_trainval,
        eval_set  = [(X_test[top_feats], Y_test)],
        callbacks = [lgb.early_stopping(50, verbose=False)],
    )
    preds = m.predict(X_test[top_feats])
    r2_by_k[K]   = r2_score(Y_test, preds)
    rmse_by_k[K] = np.sqrt(mean_squared_error(Y_test, preds))
    print(f'K={K:4d} | R²={r2_by_k[K]:.4f} | RMSE={rmse_by_k[K]:.0f} kWh'
          f' | ΔR²={r2_by_k[K] - r2_te:+.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(r2_by_k.keys()), list(r2_by_k.values()), 'o-', color='steelblue', lw=2)
axes[0].axhline(r2_te, color='red', ls='--', lw=1.2, label=f'Modèle complet R²={r2_te:.4f}')
axes[0].set(title='R² en fonction du nombre de features',
            xlabel='K (nb features)', ylabel='R² (test)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(list(rmse_by_k.keys()), list(rmse_by_k.values()), 'o-', color='coral', lw=2)
axes[1].axhline(rmse_te, color='red', ls='--', lw=1.2, label=f'Modèle complet RMSE={rmse_te:.0f} kWh')
axes[1].set(title='RMSE en fonction du nombre de features',
            xlabel='K (nb features)', ylabel='RMSE (kWh)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Feature selection — courbe du coude', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Comparaison avec lgbm_stratified (full dataset, strat composite, hyperparams manuels)
ref = {
    'label'   : 'lgbm_stratified\n(full data, composite strat)',
    'r2'      : 0.8446,
    'rmse'    : 3395.3,
    'mae'     : 1682.6,
    'rmse_pct': 28.9,
    'n'       : 549_971,
}
cur = {
    'label'   : 'lgbm_electricity_5features\n(subset filtré, 5 agrégats + non-enveloppe)',
    'r2'      : r2_te,
    'rmse'    : rmse_te,
    'mae'     : mae_te,
    'rmse_pct': rmse_te / Y_test.mean() * 100,
    'n'       : len(X_sub),
}

print('=== Comparaison électricité — lgbm_stratified vs lgbm_electricity_5features ===\n')
header = f"{'Modèle':<48} {'N':>8} {'R²':>7} {'RMSE':>9} {'RMSE%':>7} {'MAE':>9}"
print(header)
print('-' * len(header))
for m in [ref, cur]:
    print(f"{m['label'].replace(chr(10),' '):48} {m['n']:>8,} {m['r2']:>7.4f} {m['rmse']:>9.0f} {m['rmse_pct']:>6.1f}% {m['mae']:>9.0f}")

delta_r2   = cur['r2']   - ref['r2']
delta_rmse = cur['rmse'] - ref['rmse']
print(f"\nΔ R²   : {delta_r2:+.4f}  ({'amélioration' if delta_r2>0 else 'dégradation'})")
print(f"Δ RMSE : {delta_rmse:+.0f} kWh  ({'amélioration' if delta_rmse<0 else 'dégradation'})")

# ── Graphique comparatif ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
labels  = ['lgbm_stratified\n(full data)', 'lgbm_5features\n(subset filtré)']
colors  = ['#aec7e8', '#1f77b4']

for ax, metric, vals, ylabel in [
    (axes[0], 'R²',     [ref['r2'],       cur['r2']],       'R²'),
    (axes[1], 'RMSE',   [ref['rmse'],     cur['rmse']],     'RMSE (kWh)'),
    (axes[2], 'RMSE%',  [ref['rmse_pct'], cur['rmse_pct']], 'RMSE relative (%)'),
]:
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', width=0.5)
    for b, v in zip(bars, vals):
        fmt = f'{v:.4f}' if metric == 'R²' else f'{v:.1f}'
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + max(vals)*0.01,
                fmt, ha='center', fontsize=10, fontweight='bold')
    ax.set_title(metric); ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.3); ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Comparaison des modèles LightGBM — cible électricité', fontsize=12)
plt.tight_layout()
plt.show()

## Modèles par usage électrique
Un modèle par cible (`total`, `chauffage`, `clim`, `eau_chaude`), avec les mêmes features (5 agrégats + non-enveloppe), le même split stratifié ASHRAE et les hyperparamètres optimisés sur `total`.

In [ ]:
# Entraînement d'un modèle par usage (mêmes hyperparamètres Optuna, même split) + métriques
resultats  = {}
preds_test = {}
models     = {}

for nom in TARGETS:
    y_tv = Yt_trainval[nom]
    y_te = Yt_test[nom]

    m = lgb.LGBMRegressor(**final_params)
    m.fit(X_trainval, y_tv, eval_set=[(X_test, y_te)],
          callbacks=[lgb.early_stopping(100, verbose=False)])

    p = m.predict(X_test)
    r2, rmse, mae = metrics(y_te, p)
    moy = y_te.mean()
    resultats[nom] = {
        'R2': r2, 'RMSE': rmse, 'MAE': mae,
        'RMSE_%': rmse / moy * 100 if moy > 0 else np.nan,
        'moyenne': moy, 'part_>0_%': (y_te > 0).mean() * 100,
        'arbres': m.best_iteration_,
    }
    preds_test[nom] = p
    models[nom] = m
    print(f'{nom:12} | R²={r2:.4f} | RMSE={rmse:7.0f} kWh | MAE={mae:6.0f} | arbres={m.best_iteration_}')

res = pd.DataFrame(resultats).T
res = res[['R2', 'RMSE', 'RMSE_%', 'MAE', 'moyenne', 'part_>0_%', 'arbres']].round(2)
print('\n=== Résultats par usage (test) ===')
print(res.to_string())

In [ ]:
# Graphiques comparatifs par usage
ordre = res.sort_values('R2', ascending=True).index

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].barh(ordre, res.loc[ordre, 'R2'], color='steelblue', edgecolor='white')
axes[0].set(title='R² par usage', xlabel='R² (test)'); axes[0].grid(axis='x', alpha=0.3)
for i, v in enumerate(res.loc[ordre, 'R2']):
    axes[0].text(v, i, f' {v:.3f}', va='center', fontsize=9)

axes[1].barh(ordre, res.loc[ordre, 'RMSE'], color='coral', edgecolor='white')
axes[1].set(title='RMSE par usage', xlabel='RMSE (kWh)'); axes[1].grid(axis='x', alpha=0.3)
for i, v in enumerate(res.loc[ordre, 'RMSE']):
    axes[1].text(v, i, f' {v:.0f}', va='center', fontsize=9)

plt.suptitle('Performance LightGBM par usage électrique — subset filtré, 5 agrégats', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Réel vs Prédit pour chaque usage
n    = len(TARGETS)
ncol = 2
nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.5*ncol, 4.5*nrow))
axes = axes.ravel()

for ax, nom in zip(axes, TARGETS):
    yt = Yt_test[nom].values
    p  = preds_test[nom]
    ax.scatter(yt, p, s=4, alpha=0.2, color='steelblue', edgecolors='none')
    hi = max(yt.max(), p.max()) * 1.02
    ax.plot([0, hi], [0, hi], 'k--', lw=1)
    ax.text(0.04, 0.95, f"R²={resultats[nom]['R2']:.3f}", transform=ax.transAxes,
            va='top', fontsize=11, bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    ax.set(title=nom, xlabel='réel (kWh)', ylabel='prédit (kWh)')
    ax.grid(alpha=0.2)

for ax in axes[n:]:
    ax.axis('off')
plt.suptitle('Réel vs Prédit par usage électrique — 5 agrégats + non-enveloppe', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Importance des features pour un usage choisi (agrégats physiques en rouge)
usage = 'chauffage'   # <-- changer pour explorer un autre usage
TOP_N = 20            # nombre de features affichées

imp = pd.Series(models[usage].feature_importances_,
                index=X_trainval.columns).sort_values(ascending=False)
top = imp.head(TOP_N).sort_values()

labels = [name if len(name) <= 45 else name[:42] + '…' for name in top.index]
colors_u = ['crimson' if f in AGG else 'seagreen' for f in top.index]

fig, ax = plt.subplots(figsize=(11, 0.45 * TOP_N + 1))
ax.barh(range(len(top)), top.values, color=colors_u)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(labels, fontsize=9)
ax.set(title=f'Feature importance — usage « {usage} » (top {TOP_N}, rouge = agrégats physiques)',
       xlabel='Importance')
ax.grid(axis='x', alpha=0.3)
ax.margins(y=0.01)
plt.tight_layout()
plt.show()

part_agg = imp[imp.index.isin(AGG)].sum() / imp.sum() * 100
print(f'Top 15 features pour « {usage} » :')
print(imp.head(15).round(0).to_string())
print(f'\nPart cumulée des {len(AGG)} agrégats physiques pour « {usage} » : {part_agg:.1f} %')

# Part des agrégats dans chaque usage
print(f'\n=== Part des {len(AGG)} agrégats physiques par usage ===')
for nom in TARGETS:
    i = pd.Series(models[nom].feature_importances_, index=X_trainval.columns)
    print(f'  {nom:12} : {i[i.index.isin(AGG)].sum() / i.sum() * 100:5.1f} %')

In [ ]:
# ── Classement de TOUS les attributs par importance, pour chaque cible ────────
imp_usages = pd.DataFrame({nom: pd.Series(models[nom].feature_importances_, index=X_trainval.columns)
                           for nom in TARGETS})

pd.set_option('display.max_rows', None)
for nom in TARGETS:
    s = imp_usages[nom].sort_values(ascending=False)
    tot = s.sum() or 1
    tab = pd.DataFrame({
        'importance': s.round(0).astype(int),
        'part_%'    : (s / tot * 100).round(2),
        'agrégat'   : ['oui' if f in AGG else '' for f in s.index],
    })
    tab.index.name = 'feature'
    tab.insert(0, 'rang', range(1, len(tab) + 1))

    print(f"=== {nom} — {len(tab)} attributs classés par importance "
          f"(R²={resultats[nom]['R2']:.3f}) ===\n")
    print(tab.to_string())
    print(f"\nPart cumulée des 5 agrégats physiques : "
          f"{tab.loc[tab['agrégat'] == 'oui', 'part_%'].sum():.1f} %")
    print('\n' + '─' * 100 + '\n')
pd.reset_option('display.max_rows')

In [ ]:
# ── Vue comparative : rang de chaque attribut dans les 4 cibles ───────────────
comp = pd.DataFrame({f'rang_{nom}': imp_usages[nom].rank(ascending=False, method='min').astype(int)
                     for nom in TARGETS})
for nom in TARGETS:
    comp[f'part%_{nom}'] = (imp_usages[nom] / imp_usages[nom].sum() * 100).round(2)
comp['agrégat'] = ['oui' if f in AGG else '' for f in comp.index]
comp.index.name = 'feature'
comp = comp.sort_values(f'rang_{TARGET}')

cols = [f'rang_{n}' for n in TARGETS] + [f'part%_{n}' for n in TARGETS] + ['agrégat']

pd.set_option('display.max_rows', None)
print(f'=== Rang des {len(comp)} attributs dans chaque cible (trié par rang sur « {TARGET} ») ===\n')
print(comp[cols].to_string())
pd.reset_option('display.max_rows')

# ── Graphe : tous les attributs, un panneau par cible (ordre commun = importance 'total') ──
ordre_commun = imp_usages[TARGET].sort_values().index
fig, axes = plt.subplots(1, len(TARGETS), figsize=(5 * len(TARGETS), max(8, len(ordre_commun) * 0.22)),
                         sharey=True)

for ax, nom in zip(np.atleast_1d(axes), TARGETS):
    vals = (imp_usages.loc[ordre_commun, nom] / imp_usages[nom].sum() * 100)
    cols_b = ['crimson' if f in AGG else 'steelblue' for f in ordre_commun]
    ax.barh(range(len(ordre_commun)), vals.values, color=cols_b)
    ax.set(title=nom, xlabel='part de l\'importance (%)')
    ax.grid(axis='x', alpha=0.3)

np.atleast_1d(axes)[0].set_yticks(range(len(ordre_commun)))
np.atleast_1d(axes)[0].set_yticklabels(ordre_commun, fontsize=7)
plt.suptitle(f'Importance de tous les attributs par cible — ordre commun : importance « {TARGET} » '
             f'(rouge = agrégats physiques)', fontsize=13)
plt.tight_layout()
plt.show()